Projeto: Merca Data Platform

Squad: 2 | Streaming em Tempo Real



| **Objetivo** | Conectar ao ADLS Gen2 e validar estrutura do container |

| **Container** | real-time-ecommerce-data |

 | **Depende de** | feat_squad2_99_helpers |



In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
inicio = log_inicio("feat_squad2_00_setup_config")

try:
    container_client = get_container_client()
    log.info(f"Conexão com ADLS estabelecida!")
    log.info(f"Container: {ADLS_CONTAINER}")
except Exception as e:
    log.error(f"Erro ao conectar no ADLS: {str(e)}")
    raise

## 3. Listar Estrutura do Container

try:
    itens = list(container_client.get_paths())
    log.info(f"{len(itens)} item(ns) encontrado(s) no container\n")

    for item in itens:
        tipo = "Pastas" if item.is_directory else "Arquivos"
        print(f"  {tipo} {item.name}")

except Exception as e:
    log.error(f"Erro ao listar container: {str(e)}")
    raise

## 4. Validar Snapshots Disponíveis

try:
    snapshots = listar_snapshots()
    log.info(f"{len(snapshots)} snapshot(s) encontrado(s)\n")

    for snap in sorted(snapshots):
        print(f"Pacotes {snap}")

    # Snapshot mais recente
    mais_recente = get_snapshot_mais_recente()
    log.info(f"Snapshot mais recente: {mais_recente}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise

## 5. Validar Tabelas do Squad 2

try:
    snap_ref = get_snapshot_mais_recente()
    log.info(f"Validando tabelas no snapshot: {snap_ref}\n")

    for tabela in TABELAS_SQUAD2:
        try:
            df = ler_parquet(snap_ref, tabela)
            log.info(
                f"OK {tabela} → "
                f"{df.count()} linhas | "
                f"{len(df.columns)} colunas"
            )
        except Exception as e:
            log.error(f"Erro em {tabela}: {str(e)}")

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

log_fim("feat_squad2_00_setup_config", inicio)